# SUBSIDE Science Backbone — Setup & Preprocessing

**Companion to:** `SUBSIDE_Science_Backbone.ipynb` (the main analysis notebook).

This notebook produces a **referenced, reusable science backbone** — a JSON file
the main notebook loads as **Tier 0**, ahead of the ETO/sbp/inlined waterfall it
already supports. Build it once, commit it alongside the corpus, and every
downstream analysis is anchored on the same, citable reference frame.

### Why a separate setup step?

Two ideas from the scientometrics literature motivate the split:

1. **A science basemap is a reference system, not a result.** Börner, Klavans,
   Boyack and colleagues frame the UCSD Map of Science as the cartographic
   analogue of Mercator's world map — a stable scaffold on which heterogeneous
   *data overlays* are placed [1, 2]. Treating the backbone as a setup artifact
   keeps it out of the experiment loop.
2. **A consensus map outperforms any single map.** Klavans & Boyack (2009)
   showed that pooling 20 distinct global maps of science yields a more stable
   ordering of disciplines than any one source [3]. The same logic applies
   locally: combining the **paper-level, current ETO Map of Science** [4] with
   the **journal-level UCSD canonical 13-discipline scaffold** [1] produces a
   backbone more robust than either alone.

### Three layers, decreasing verification → increasing locality

| Layer | Source | What it is | When to use |
|---|---|---|---|
| **A** | [ETO Map of Science](https://sciencemap.eto.tech/) cluster CSV export | Paper-level Leiden clusters over the Merged Academic Corpus (~92,000 clusters), Klavans/Boyack lineage methodology updated to OpenAlex/Semantic Scholar/Web of Science [4] | Most current, paper-level granularity. Requires one manual export step. |
| **B** | UCSD Map of Science 13-discipline / 554-subdiscipline classification [1] | Journal-level, published 2012 (10-year update covering 2001–2010, ~25,000 journals), CC BY-NC-SA 3.0 | Stable, citable canonical scaffold. Use when ETO is unavailable or when you want a cross-corpus-comparable reference frame. |
| **C** | SUBSIDE-tuned inline fallback (lives in `SUBSIDE_Science_Backbone.ipynb` §5) | Curated for subsidence vocabulary | Local, fast-start, intentionally narrow — for early iteration only. |

The output of this notebook is a single JSON file (`science_backbone.json`) that
the main notebook reads via a Tier-0 hook. **You can re-run the main notebook
many times without re-running this one.**

### References

1. Börner K, Klavans R, Patek M, Zoss AM, Biberstine JR, Light RP, Larivière V, Boyack KW (2012). *Design and Update of a Classification System: The UCSD Map of Science.* PLOS ONE 7(7): e39464. https://doi.org/10.1371/journal.pone.0039464
2. Shiffrin RM, Börner K (2004). *Mapping Knowledge Domains.* PNAS 101 (Suppl. 1): 5183–5185. https://doi.org/10.1073/pnas.0307852100
3. Klavans R, Boyack KW (2009). *Toward a Consensus Map of Science.* JASIST 60(3): 455–476. https://doi.org/10.1002/asi.20991
4. Emerging Technology Observatory, Center for Security and Emerging Technology, Georgetown University. *ETO Map of Science.* https://sciencemap.eto.tech/ — methodology at https://eto.tech/dataset-docs/mac-clusters/

---

## 1. Setup

Paths, imports, and the BibTeX corpus loader from the main notebook.


In [7]:
from pathlib import Path
import os, re, json
from datetime import datetime

import numpy as np
import pandas as pd

# ── Paths (mirror SUBSIDE_Science_Backbone.ipynb §1.1) ───────────────
CANDIDATE_BIB_PATHS = [
    Path("/work/01813/sawp33/MySUBSIDE/Global_SUBSIDE_2024.bib"),
    Path(os.getcwd()) / "Global_SUBSIDE_2024.bib",
    Path(os.getcwd()) / "data" / "Global_SUBSIDE_2024.bib",
]
BIB_PATH = next((p for p in CANDIDATE_BIB_PATHS if p.exists()),
                CANDIDATE_BIB_PATHS[0])

OUTPUT_DIR = Path(os.getcwd()) / "ScienceBackboneResults"
OUTPUT_DIR.mkdir(exist_ok=True)

# Where the main notebook will look for the produced backbone JSON
BACKBONE_JSON_PATH = OUTPUT_DIR / "science_backbone.json"

# If you've already exported an ETO Map of Science CSV, point at it here.
# Leave None to skip Layer A entirely.
import os

ETO_CSV_PATH = Path(os.environ["WORK"]) / "dso_cookbook_fixes" / "eto-map-of-scienceWaterResources.csv"

print(f"BibTeX source : {BIB_PATH}  (exists = {BIB_PATH.exists()})")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Backbone JSON : {BACKBONE_JSON_PATH}")
print(f"ETO CSV path  : {ETO_CSV_PATH}")


BibTeX source : /work/01813/sawp33/MySUBSIDE/Global_SUBSIDE_2024.bib  (exists = True)
Output dir    : /scratch/01813/sawp33/tapis/b673a5f6-e828-4ff9-add8-fd125f967b4a-007/work/dso_cookbook_fixes/ScienceBackboneResults
Backbone JSON : /scratch/01813/sawp33/tapis/b673a5f6-e828-4ff9-add8-fd125f967b4a-007/work/dso_cookbook_fixes/ScienceBackboneResults/science_backbone.json
ETO CSV path  : /work/01813/sawp33/ls6/dso_cookbook_fixes/eto-map-of-scienceWaterResources.csv


In [8]:
# Dependency check — same set as the main notebook
import sys, subprocess, importlib

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _p, _m in [("bibtexparser", "bibtexparser"),
               ("scikit-learn", "sklearn"),
               ("plotly", "plotly")]:
    _ensure(_p, _m)

import bibtexparser
from bibtexparser.bparser import BibTexParser
from sklearn.feature_extraction.text import TfidfVectorizer

print("✓ Imports complete")


✓ Imports complete


## 2. Summarize your corpus for ETO query construction

Before going to the ETO Map UI, we need a short list of distinctive search
terms that describe the corpus well. These are the words you'll paste into
the Map's search bar so that the cluster export covers the relevant slice of
science instead of all 92,000 clusters.


In [9]:
def load_bib_quick(path):
    '''Minimal BibTeX loader — just enough to harvest keywords/abstracts.'''
    parser = BibTexParser(common_strings=True)
    parser.ignore_nonstandard_types = False
    parser.homogenize_fields = True
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        return bibtexparser.load(fh, parser=parser).entries


def _clean(text):
    if not text:
        return ""
    t = re.sub(r"[{}\\]", " ", str(text))
    return re.sub(r"\s+", " ", t).strip()


if BIB_PATH.exists():
    raw = load_bib_quick(BIB_PATH)
    rows = []
    for e in raw:
        rows.append({
            "title":    _clean(e.get("title")),
            "abstract": _clean(e.get("abstract")),
            "keywords": _clean(e.get("keywords")),
        })
    df_quick = pd.DataFrame(rows)
    df_quick["text_content"] = (df_quick["title"].fillna("") + " . " +
                                df_quick["abstract"].fillna("") + " . " +
                                df_quick["keywords"].fillna(""))
    df_quick = df_quick[df_quick["text_content"].str.strip().str.len() > 0]
    print(f"✓ Loaded {len(df_quick):,} entries from {BIB_PATH.name}")
else:
    df_quick = pd.DataFrame(columns=["title","abstract","keywords","text_content"])
    print(f"  ⚠ {BIB_PATH} not found — Section 4 below will still work but with empty input")


✓ Loaded 8,409 entries from Global_SUBSIDE_2024.bib


### 2.1 Top distinctive terms

A short, ranked list of bigrams + unigrams that distinguish this corpus. Use
the top 10–20 as your ETO search seeds.


In [10]:
ETO_SEARCH_SEEDS_N = 25

if len(df_quick):
    vec = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.7,
        stop_words="english",
    )
    X = vec.fit_transform(df_quick["text_content"].tolist())
    means = np.asarray(X.mean(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    order = means.argsort()[::-1]
    # Filter out pure digits / 1-char tokens
    seeds = [vocab[i] for i in order
             if not vocab[i].isdigit() and len(vocab[i]) > 2][:ETO_SEARCH_SEEDS_N]
    print(f"Top {len(seeds)} distinctive terms (paste into the ETO Map search bar):\n")
    for i, s in enumerate(seeds, 1):
        print(f"  {i:2d}. {s}")
else:
    seeds = []
    print("(no entries — skipping)")


Top 25 distinctive terms (paste into the ETO Map search bar):

   1. groundwater
   2. land
   3. water
   4. land subsidence
   5. level
   6. mining
   7. surface
   8. deformation
   9. area
  10. sea
  11. basin
  12. data
  13. model
  14. study
  15. ground
  16. coastal
  17. areas
  18. soil
  19. results
  20. nan
  21. sea level
  22. using
  23. high
  24. coal
  25. analysis


## 3. Export the ETO Map of Science cluster slice (Layer A)

The ETO Map's cluster metadata isn't a public bulk download — the underlying
Research Cluster Dataset has commercial-license restrictions from Clarivate
and others. But the Map UI lets you **build a query, switch to list view,
and download the matched clusters as CSV** for any analytic environment. That
CSV is what we ingest here.

> **Per ETO's terms of use:** when you cite work derived from the Map of Science,
> attribute it as `"ETO Map of Science"` and link to `https://sciencemap.eto.tech/`.

### Click-by-click steps

1. **Open** [https://sciencemap.eto.tech/](https://sciencemap.eto.tech/) in a browser.
2. **Search** using a few of the distinctive terms you saw in §2.1
   (e.g. *land subsidence*, *insar*, *groundwater*). The search box is at the top
   of the left-hand filter panel.
3. **Refine with filters** if you want — country, growth rating, subject
   filters in the left sidebar. For a literature-scoping run you usually want
   to leave growth/citation filters wide open.
4. **Switch to list view** using the toggle near the top of the results.
5. **Customize columns**. The columns you'll want for the SUBSIDE backbone are:
   `cluster_id`, `cluster_title`, `cluster_summary` (or `summary`),
   `disciplines` (3 top-scoring), `fields` (3 top-scoring), `subfields`,
   `topics`, `key_concepts`, `map_x` / `map_y` if available, `size`.
6. **Download CSV** — the export button is in the list-view header. Save the
   file somewhere this notebook can read (e.g. next to your `.bib`).
7. **Set `ETO_CSV_PATH`** in §1 above to that file's path, then re-run.

Layers B (UCSD canonical) and C (SUBSIDE inline) continue to work even if you
skip the ETO export — Tier 0 will pick up whichever layer(s) you've populated.

The ETO export comes from a Leiden-clustered network that combines between-article
**citations** with text-embedding similarity (multilingual SentenceTransformer),
a contemporary descendant of the bibliographic coupling and co-citation
techniques pioneered by Boyack, Klavans, and Börner.


### 3.1 Ingest the ETO cluster CSV

Tolerant loader: the column names vary somewhat across Map-of-Science updates,
so we look for the *shape* of each field rather than relying on exact names.


In [11]:
#PATCH FOR 3.1 and 3.2 cells below
"""
Drop-in replacement for §3.1 and §3.2 of SUBSIDE_ScienceBackbone_Setup.ipynb.

Handles the May-2026 ETO Map of Science export format, where the previous
hierarchical disciplines/fields/topics columns have been consolidated into
a single 'Most common research field' column with 12 top-level values.

Three configurable knobs near the top:
  ETO_TITLE_FILTER       — list of patterns; keep clusters mentioning any of them
  ETO_FIELD_WHITELIST    — restrict to a subset of ETO's 12 fields (None = all)
  N_SUBS_PER_DOMAIN      — cap on subdisciplines per UCSD-mapped domain
"""

from collections import defaultdict, Counter

# ─────────────────────────────────────────────────────────────────────
# §3.1 — Ingest the ETO cluster CSV (new May 2026 single-field format)
# ─────────────────────────────────────────────────────────────────────

# After _norm() strips whitespace/underscore/dash, 'Most common research field'
# normalizes to 'mostcommonresearchfield' — we recognize that and a few others.
COL_CANDIDATES = {
    "cluster_id": ["clusterid", "id", "cluster"],
    "title":      ["clustertitle", "title", "name"],
    "summary":    ["clustersummary", "summary", "description"],
    "field":      ["mostcommonresearchfield", "mostcommonfield", "researchfield",
                   "field", "disciplines", "discipline"],
    "size":       ["clustersize", "size", "narticles", "articlecount"],
    "growth":     ["growthrating", "growth"],
    "citation":   ["citationrating", "citation"],
}

# ETO's 12 top-level fields → UCSD's 13 canonical disciplines.
# A few collapses are unavoidable (medicine spans 4 UCSD disciplines, materials
# science could go to either Chemistry or CMCE) — we pick the closest single
# UCSD bucket so the Layer A → Layer B merge in §5 is unambiguous.
ETO_FIELD_TO_UCSD = {
    "earth science":      "Earth Sciences",
    "biology":            "Biology",
    "chemistry":          "Chemistry",
    "materials science":  "Chemistry",
    "computer science":   "Electrical Engineering & Computer Science",
    "engineering":        "Chemical, Mechanical, & Civil Engineering",
    "mathematics":        "Math & Physics",
    "physics":            "Math & Physics",
    "medicine":           "Medical Specialties",
    "humanities":         "Humanities",
    "social science":     "Social Sciences",
    "business":           "Social Sciences",
}

# SUBSIDE-relevant cluster filter. The full export is 91,585 clusters spanning
# all of science; we keep only those whose title or summary mentions at least
# one of these terms. Edit freely. Set ETO_TITLE_FILTER = None to keep all.
ETO_TITLE_FILTER = [
    "subsidence", "compaction", "consolidation", "uplift", "rebound", "heave",
    "groundwater", "aquifer", "hydrolog", "watershed", "phreatic",
    "oil", "petroleum", "reservoir", "co2 storage", "sequestration",
    "mining", "longwall", "coal mine",
    "tectonic", "fault", "seismic", "geodet",
    "sediment", "delta", "alluvial",
    "insar", "interferometric", "gnss", "lidar", "remote sens",
    "coastal", "shoreline", "tidal", "estuar", "sea level", "sea-level", "marsh",
    "flood", "inundation", "storm surge",
    "modflow", "groundwater model",
    "land use", "land subsidence", "urban infrastructure",
    "water resource", "water management",
]
# Optional second filter: restrict to a subset of ETO's 12 fields.
# Helpful if you want only earth/engineering/materials clusters regardless of title.
ETO_FIELD_WHITELIST = None  # e.g. ["earth science", "engineering", "materials science"]


eto_clusters = None
if ETO_CSV_PATH and Path(ETO_CSV_PATH).exists():
    try:
        eto_raw = pd.read_csv(ETO_CSV_PATH)
        cols = resolve_columns(eto_raw)
        print(f"  Loaded {len(eto_raw):,} rows from {ETO_CSV_PATH}")
        print(f"  Resolved columns: {cols}")

        n_orig = len(eto_raw)

        # 1. Optional field whitelist
        if ETO_FIELD_WHITELIST and "field" in cols:
            wl = {f.lower() for f in ETO_FIELD_WHITELIST}
            mask = eto_raw[cols["field"]].astype(str).str.lower().isin(wl)
            eto_raw = eto_raw[mask]
            print(f"  Field whitelist:  {n_orig:,} → {len(eto_raw):,}")

        # 2. Optional title/summary filter
        if ETO_TITLE_FILTER and "title" in cols:
            pattern = "|".join(re.escape(t) for t in ETO_TITLE_FILTER)
            search_text = eto_raw[cols["title"]].fillna("").astype(str)
            if "summary" in cols:
                search_text = (search_text + " " +
                               eto_raw[cols["summary"]].fillna("").astype(str))
            mask = search_text.str.lower().str.contains(pattern, regex=True, na=False)
            n_pre = len(eto_raw)
            eto_raw = eto_raw[mask]
            print(f"  Title/summary filter: {n_pre:,} → {len(eto_raw):,}")

        # 3. Normalize to dicts
        recs = []
        for idx, row in eto_raw.iterrows():
            rec = {"raw_row_index": int(idx)}
            for canon, actual in cols.items():
                val = row[actual]
                rec[canon] = val if pd.notna(val) else None
            recs.append(rec)
        eto_clusters = recs

        print(f"  ✓ Normalized {len(eto_clusters):,} ETO clusters after filtering")
        if eto_clusters:
            fc = Counter(c.get("field") for c in eto_clusters).most_common()
            print(f"\n  Field distribution after filter (→ UCSD mapping):")
            for f, n in fc:
                ucsd = ETO_FIELD_TO_UCSD.get(str(f).lower(), "(unmapped)")
                print(f"    {str(f):24s} {n:>5,}  → {ucsd}")
    except Exception as e:
        print(f"  ✗ Failed to load ETO CSV: {type(e).__name__}: {e}")
elif ETO_CSV_PATH:
    print(f"  ⚠ ETO_CSV_PATH set but file not found: {ETO_CSV_PATH}")
else:
    print("  (ETO_CSV_PATH not set — skipping Layer A; using Layer B alone)")


# ─────────────────────────────────────────────────────────────────────
# §3.2 — Build Layer A from the normalized clusters
# ─────────────────────────────────────────────────────────────────────

N_SUBS_PER_DOMAIN = 25   # cap; the largest clusters per domain become subs
N_TERMS_PER_DOMAIN = 80


def build_layer_A_backbone(clusters):
    """Build Layer A from ETO clusters in the May-2026 single-field format.

    Strategy:
      - Group clusters by ETO field, then map field → UCSD canonical discipline.
      - Within each domain, sort clusters by article count (size) descending.
      - Take the top-N cluster titles as the domain's subdisciplines.
      - Seed terms from cluster title + summary tokens.
    """
    if not clusters:
        return None

    LABEL_STOP = {"the","and","of","in","on","for","to","with","by","from","an","a",
                  "or","general","other","sciences","science","research","studies",
                  "based","using","via","application","applications","analysis"}

    by_domain = defaultdict(list)
    for c in clusters:
        field = str(c.get("field") or "").lower().strip()
        if not field:
            continue
        ucsd = ETO_FIELD_TO_UCSD.get(field)
        if not ucsd:
            # Surface unmapped fields rather than silently dropping
            ucsd = f"ETO:{field.title()}"
        by_domain[ucsd].append(c)

    if not by_domain:
        return None

    backbone = {}
    for ucsd, members in by_domain.items():
        members_sorted = sorted(members,
                                key=lambda c: -float(c.get("size") or 0))
        subs = []
        seen = set()
        terms = set()
        for c in members_sorted[:N_SUBS_PER_DOMAIN]:
            title = str(c.get("title") or "").strip()
            if title and title.lower() not in seen:
                subs.append(title)
                seen.add(title.lower())
            text = (str(c.get("title") or "") + " " +
                    str(c.get("summary") or "")).lower()
            for w in re.findall(r"[a-z][a-z\-]+", text):
                if len(w) > 3 and w not in LABEL_STOP:
                    terms.add(w)
        backbone[ucsd] = {
            "subdisciplines": subs or ["General"],
            "terms":          sorted(terms)[:N_TERMS_PER_DOMAIN] or [ucsd.lower()],
            "_source":        (f"ETO Map of Science ({len(members)} clusters; "
                               f"primary field='{members[0].get('field')}')"),
            "_n_clusters":    len(members),
        }
    return backbone


layer_A = build_layer_A_backbone(eto_clusters) if eto_clusters else None
if layer_A:
    print(f"✓ Layer A built: {len(layer_A)} domains from ETO")
    for d, node in sorted(layer_A.items(),
                          key=lambda x: -x[1].get("_n_clusters", 0)):
        print(f"   · {d:55s} ({node['_n_clusters']:>4} clusters, "
              f"{len(node['subdisciplines']):>2} subs, "
              f"{len(node['terms']):>3} terms)")
else:
    print("(Layer A: empty — no clusters mapped)")


  ✗ Failed to load ETO CSV: NameError: name 'resolve_columns' is not defined
(Layer A: empty — no clusters mapped)


REPLACED
# Canonical column name candidates (lower-case, comparison ignores spaces/_/-)
COL_CANDIDATES = {
    "cluster_id":      ["clusterid", "id", "cluster"],
    "title":           ["clustertitle", "title", "name"],
    "summary":         ["clustersummary", "summary", "description"],
    "disciplines":     ["disciplines", "discipline", "researchdisciplines"],
    "fields":          ["fields", "field", "researchfields"],
    "subfields":       ["subfields", "subfield"],
    "topics":          ["topics", "topic"],
    "key_concepts":    ["keyconcepts", "concepts", "keywords"],
    "map_x":           ["mapx", "x", "xcoord"],
    "map_y":           ["mapy", "y", "ycoord"],
    "size":            ["size", "narticles", "articlecount"],
}

def _norm(s):
    return re.sub(r"[\s_\-]+", "", str(s).strip().lower())


def resolve_columns(eto_df):
    '''Return a dict mapping canonical names → actual column names in eto_df.'''
    avail = {_norm(c): c for c in eto_df.columns}
    resolved = {}
    for canon, candidates in COL_CANDIDATES.items():
        for c in candidates:
            if c in avail:
                resolved[canon] = avail[c]
                break
    return resolved


def _split_multivalue(cell):
    '''ETO often stores multi-valued columns as 'A | B | C' or 'A; B; C' or JSON list.'''
    if cell is None or (isinstance(cell, float) and np.isnan(cell)):
        return []
    s = str(cell).strip()
    if not s or s.lower() in {"nan", "none", "[]"}:
        return []
    # Try JSON list
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except Exception:
            pass
    # Fall back to delimiter split
    parts = re.split(r"\s*[\|;]\s*", s)
    if len(parts) == 1:
        # Try comma when there's no pipe / semicolon
        parts = re.split(r"\s*,\s*", s)
    return [p.strip() for p in parts if p.strip()]


eto_clusters = None
if ETO_CSV_PATH and Path(ETO_CSV_PATH).exists():
    try:
        eto_raw = pd.read_csv(ETO_CSV_PATH)
        cols = resolve_columns(eto_raw)
        print(f"  Loaded {len(eto_raw):,} rows from {ETO_CSV_PATH}")
        print(f"  Resolved columns: {cols}")
        # Normalize: every row becomes a dict with canonical keys
        recs = []
        for _, row in eto_raw.iterrows():
            rec = {"raw_row_index": _}
            for canon, actual in cols.items():
                val = row[actual]
                if canon in ("disciplines","fields","subfields","topics","key_concepts"):
                    rec[canon] = _split_multivalue(val)
                else:
                    rec[canon] = val if pd.notna(val) else None
            recs.append(rec)
        eto_clusters = recs
        print(f"  ✓ Normalized {len(eto_clusters):,} ETO clusters")
    except Exception as e:
        print(f"  ✗ Failed to load ETO CSV: {type(e).__name__}: {e}")
elif ETO_CSV_PATH:
    print(f"  ⚠ ETO_CSV_PATH set but file not found: {ETO_CSV_PATH}")
else:
    print("  (ETO_CSV_PATH not set — skipping Layer A; using Layer B alone)")


### 3.2 Build the Layer-A backbone from ETO clusters

ETO discipline → subdiscipline mapping is built bottom-up: each unique
`discipline` value becomes a domain, the `fields` it appears with become
subdisciplines, and the union of `key_concepts` + `topics` becomes the
keyword/term set. Coordinates are dropped (the main notebook computes its
own spring layout from the dict structure).


REPLACED
def build_layer_A_backbone(clusters):
    '''Return a dict matching the main notebook's SCIENCE_BACKBONE shape:
        {domain_name: {"subdisciplines": [...], "terms": [...]}}
    '''
    if not clusters:
        return None

    domain_subs = {}    # domain → set of subdisciplines (fields)
    domain_terms = {}   # domain → set of terms (key_concepts + topics)

    for c in clusters:
        disciplines = c.get("disciplines") or []
        fields      = c.get("fields") or []
        topics      = c.get("topics") or []
        concepts    = c.get("key_concepts") or []
        title       = c.get("title")

        if not disciplines:
            continue
        # Each cluster usually has its top 3 disciplines & top 3 fields.
        # We use the *primary* discipline as the parent.
        primary_d = disciplines[0]
        domain_subs.setdefault(primary_d, set())
        domain_terms.setdefault(primary_d, set())

        for f in fields:
            domain_subs[primary_d].add(f)
        for t in topics + concepts:
            if 2 <= len(t) <= 60:
                domain_terms[primary_d].add(t.lower())
        # Cluster titles are often informative short labels — include them as terms
        if title and 2 <= len(str(title)) <= 80:
            domain_terms[primary_d].add(str(title).lower())

    backbone = {}
    for d, subs in domain_subs.items():
        terms = sorted(domain_terms.get(d, set()))
        backbone[d] = {
            "subdisciplines": sorted(subs)[:12] or ["General"],
            "terms":          terms[:50] or [d.lower()],
            "_source":        "ETO Map of Science cluster CSV export",
            "_n_clusters":    sum(1 for c in clusters
                                  if (c.get("disciplines") or [None])[0] == d),
        }
    return backbone


layer_A = build_layer_A_backbone(eto_clusters) if eto_clusters else None
if layer_A:
    print(f"✓ Layer A built: {len(layer_A)} domains from ETO")
    for d, node in layer_A.items():
        print(f"   · {d:40s} ({node['_n_clusters']} clusters, "
              f"{len(node['subdisciplines'])} subdisciplines, "
              f"{len(node['terms'])} terms)")
else:
    print("(Layer A: empty — ETO CSV not loaded)")


## 4. UCSD canonical 13-discipline scaffold (Layer B)

The 2010 UCSD Map of Science classifies ~25,000 journals into **554
subdisciplines** aggregated into **13 disciplines** [Börner et al. 2012].
Each subdiscipline has its own keyword set and x/y coordinates from the
spherical layout. The full 554-subdiscipline classification ships with the
paper's supplementary data (CC BY-NC-SA 3.0) and is also redistributed as a
Pajek `.net` file by the Science Integrity Alliance replication repo.

Here we hardcode:
- the **13 canonical discipline names** with their published colors, and
- a **representative set of subdisciplines per discipline** sufficient to
  anchor cross-disciplinary work (especially for subsidence, geosciences,
  remote sensing, water resources, infrastructure, policy).

The seed terms per subdiscipline are deliberately compact — extend them as
your corpus reveals new vocabulary.

> **Citation requirement.** UCSD Map of Science is CC BY-NC-SA 3.0. When you
> use this scaffold downstream, cite Börner et al. 2012 (PLOS ONE) and include
> the standard attribution to The Regents of the University of California,
> SciTech Strategies, Observatoire des Sciences et des Technologies, and the
> Cyberinfrastructure for Network Science Center.


In [12]:
# ── UCSD 13-discipline canonical scaffold ────────────────────────────
# Discipline names and color palette per published topical visualizations
# (CNS-IU, VIVO) of the 2010 UCSD map.
#
# Each discipline ships with a representative set of subdisciplines + seed
# terms. The full 554-subdiscipline list lives in the published supplement;
# extend below as your corpus calls for it.

UCSD_SCAFFOLD = {
    "Math & Physics": {
        "color": "#7570B3",
        "subdisciplines": ["Mathematics", "Applied Mathematics", "Statistics",
                           "Theoretical Physics", "Condensed Matter Physics",
                           "Astrophysics", "Geodesy", "Geophysics"],
        "terms": ["mathematics","statistics","probability","physics","quantum",
                  "relativity","astrophysics","cosmology","geodesy","gravity",
                  "geophysics","seismic","numerical","tensor","equation"],
    },
    "Chemistry": {
        "color": "#1F78B4",
        "subdisciplines": ["Analytical Chemistry", "Organic Chemistry",
                           "Inorganic Chemistry", "Physical Chemistry",
                           "Polymer Chemistry", "Geochemistry"],
        "terms": ["chemistry","chemical","organic","inorganic","catalysis",
                  "polymer","molecule","reaction","spectroscopy","geochemistry",
                  "isotope"],
    },
    "Earth Sciences": {
        "color": "#B15928",
        "subdisciplines": ["Hydrology", "Hydrogeology", "Geomorphology",
                           "Sedimentology", "Tectonics", "Climatology",
                           "Oceanography", "Soil Science", "Remote Sensing",
                           "Land Subsidence", "Coastal Science"],
        "terms": ["earth","geology","geological","hydrology","groundwater",
                  "aquifer","sediment","sedimentation","tectonic","fault",
                  "subsidence","compaction","consolidation","coastal","delta",
                  "shoreline","sea level","sea-level","insar","interferometric",
                  "remote sensing","sentinel","gps","gnss","lidar","leveling",
                  "extensometer","soil","watershed","climate","precipitation",
                  "drought","oceanography","seismic"],
    },
    "Biology": {
        "color": "#33A02C",
        "subdisciplines": ["Ecology", "Botany", "Zoology",
                           "Evolutionary Biology", "Marine Biology",
                           "Microbiology", "Plant Biology"],
        "terms": ["biology","ecology","ecosystem","habitat","species",
                  "biodiversity","wetland","marsh","mangrove","plant",
                  "vegetation","microbial","organism","population"],
    },
    "Biotechnology": {
        "color": "#FB9A99",
        "subdisciplines": ["Genetics", "Genomics", "Molecular Biology",
                           "Biochemistry", "Bioengineering", "Synthetic Biology"],
        "terms": ["biotechnology","gene","genetic","genome","protein","enzyme",
                  "molecular","biochemistry","bioengineering","synthetic"],
    },
    "Infectious Disease": {
        "color": "#A6CEE3",
        "subdisciplines": ["Virology", "Immunology", "Parasitology",
                           "Epidemiology", "Public Health Microbiology"],
        "terms": ["infection","virus","viral","immunology","bacteria",
                  "pathogen","epidemic","epidemiology","outbreak","vaccine"],
    },
    "Medical Specialties": {
        "color": "#E31A1C",
        "subdisciplines": ["Cardiology", "Oncology", "Neurology", "Surgery",
                           "Radiology", "Internal Medicine"],
        "terms": ["clinical","patient","medicine","cardiology","cancer",
                  "oncology","surgery","radiology","diagnosis","treatment"],
    },
    "Health Professionals": {
        "color": "#FDBF6F",
        "subdisciplines": ["Public Health", "Nursing", "Health Services Research",
                           "Health Policy", "Environmental Health"],
        "terms": ["public health","nursing","health services","health policy",
                  "environmental health","occupational","wellbeing","care"],
    },
    "Brain Research": {
        "color": "#FF7F00",
        "subdisciplines": ["Neuroscience", "Cognitive Science", "Psychiatry",
                           "Behavioral Neuroscience"],
        "terms": ["brain","neural","neuron","cognition","cognitive","memory",
                  "behavior","psychiatry","neuroscience"],
    },
    "Electrical Engineering & Computer Science": {
        "color": "#CAB2D6",
        "subdisciplines": ["Computer Science", "Machine Learning",
                           "Artificial Intelligence", "Signal Processing",
                           "Electrical Engineering", "Computer Vision",
                           "Networks & Distributed Systems"],
        "terms": ["algorithm","computer","computing","machine learning",
                  "deep learning","neural network","convolutional","lstm",
                  "signal processing","electrical","gis","spatial analysis",
                  "computer vision","data science","artificial intelligence"],
    },
    "Chemical, Mechanical, & Civil Engineering": {
        "color": "#6A3D9A",
        "subdisciplines": ["Civil Engineering", "Geotechnical Engineering",
                           "Hydraulic Engineering", "Structural Engineering",
                           "Mechanical Engineering", "Chemical Engineering",
                           "Petroleum Engineering", "Mining Engineering"],
        "terms": ["civil engineering","geotechnical","hydraulic","structural",
                  "foundation","soil mechanics","mechanical","chemical engineering",
                  "petroleum","reservoir","oil","gas","co2 storage","sequestration",
                  "mining","coal","longwall","material","stress","strain",
                  "finite element"],
    },
    "Social Sciences": {
        "color": "#FFFF99",
        "subdisciplines": ["Economics", "Political Science", "Sociology",
                           "Geography", "Anthropology", "Public Policy",
                           "Decision Science"],
        "terms": ["economics","economic","political","policy","governance",
                  "regulation","stakeholder","decision support","management",
                  "sociology","geography","anthropology","community",
                  "groundwater management","gma"],
    },
    "Humanities": {
        "color": "#FFD92F",
        "subdisciplines": ["History", "Philosophy", "Literature", "Linguistics",
                           "Ethics", "Science & Technology Studies"],
        "terms": ["history","philosophy","ethics","literature","linguistics",
                  "discourse","narrative","cultural"],
    },
}

# Compact UCSD reference frame as a backbone dict
def materialize_UCSD(scaffold):
    backbone = {}
    for d, node in scaffold.items():
        backbone[d] = {
            "subdisciplines": list(node["subdisciplines"]),
            "terms":          list(node["terms"]),
            "color":          node.get("color"),
            "_source":        "UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-SA 3.0)",
        }
    return backbone

layer_B = materialize_UCSD(UCSD_SCAFFOLD)
print(f"✓ Layer B built: {len(layer_B)} disciplines (UCSD canonical)")
for d, node in layer_B.items():
    print(f"   · {d:50s} {len(node['subdisciplines']):2d} subs · "
          f"{len(node['terms']):3d} terms · {node['color']}")


✓ Layer B built: 13 disciplines (UCSD canonical)
   · Math & Physics                                      8 subs ·  15 terms · #7570B3
   · Chemistry                                           6 subs ·  11 terms · #1F78B4
   · Earth Sciences                                     11 subs ·  34 terms · #B15928
   · Biology                                             7 subs ·  14 terms · #33A02C
   · Biotechnology                                       6 subs ·  10 terms · #FB9A99
   · Infectious Disease                                  5 subs ·  10 terms · #A6CEE3
   · Medical Specialties                                 6 subs ·  10 terms · #E31A1C
   · Health Professionals                                5 subs ·   8 terms · #FDBF6F
   · Brain Research                                      4 subs ·   9 terms · #FF7F00
   · Electrical Engineering & Computer Science           7 subs ·  15 terms · #CAB2D6
   · Chemical, Mechanical, & Civil Engineering           8 subs ·  21 terms · #6A3D9A
   · 

### 4.1 Optional: load the full 554-subdiscipline UCSD classification

The compact scaffold above is one curated subdiscipline-per-row per discipline.
If you want the **full grain** — every one of the 554 published subdisciplines
with its x/y coordinates from the spherical layout — this cell fetches the
Pajek `.net` file from the Science Integrity Alliance replication repository
[5] (which redistributes the UCSD classification under CC BY-NC-SA 3.0) and
replaces `layer_B` with the full-detail version.

The parser groups subdisciplines under their parent discipline using the
`ic` (interior color) attribute, which is the discipline tag in the Pajek
file. Each subdiscipline's term list is seeded from its own label (e.g.
`"Clinical Cancer Research"` → `["clinical", "cancer", "research", "clinical cancer", ...]`)
so the full set is immediately usable for the downstream keyword-matcher in
the main notebook. Extend the term lists further as your corpus surfaces new
vocabulary.

The cell **degrades gracefully**: if the network fetch fails (firewall,
offline, etc.) or the file isn't already cached locally, it leaves the
compact scaffold from §4 in place and prints a clear "skipped" message.

**[5]** Science Integrity Alliance, *science-map* (GitHub).
https://github.com/Science-Integrity-Alliance/science-map · CC BY-NC-SA 3.0,
redistributing the 2010 UCSD Map of Science classification (Börner et al. 2012).


In [13]:
# ── Switch this off to keep the compact scaffold from §4 ─────────────
USE_UCSD_FULL = True

# Pajek .net file: 567 vertices (554 subdisciplines + 13 disciplines)
UCSD_NET_URL    = ("https://raw.githubusercontent.com/Science-Integrity-Alliance/"
                   "science-map/main/UCSDmap_with_disciplines.net.txt")
UCSD_NET_LOCAL  = OUTPUT_DIR / "UCSDmap_with_disciplines.net.txt"

# Canonical discipline labels we expect to find as named nodes in the .net
UCSD_DISCIPLINE_NAMES = {
    "Math & Physics", "Chemistry", "Earth Sciences", "Biology",
    "Biotechnology", "Infectious Disease", "Medical Specialties",
    "Health Professionals", "Brain Research",
    "Electrical Engineering & Computer Science",
    "Chemical, Mechanical, & Civil Engineering",
    "Social Sciences", "Humanities",
    # Known minor variants seen in some versions of the file:
    "Math and Physics", "Medical specialties", "Health professionals",
}

_VERTEX_RE = re.compile(
    r'^(\d+)\s+"([^"]+)"\s+([\d.eE+\-]+)\s+([\d.eE+\-]+)'
    r'(?:\s+x_fact\s+([\d.eE+\-]+))?'
    r'(?:\s+y_fact\s+([\d.eE+\-]+))?'
    r'(?:\s+ic\s+(\S+))?'
    r'(?:\s+bc\s+(\S+))?'
)


def fetch_ucsd_net(url=UCSD_NET_URL, dest=UCSD_NET_LOCAL, timeout=30):
    '''Cache to dest; only re-download if dest is missing or trivially small.'''
    # Real file is ~103 KB; 5 KB threshold catches truncated downloads while
    # allowing legitimate hand-curated subsets (or this notebook's smoke tests).
    if dest.exists() and dest.stat().st_size > 5_000:
        return dest
    import urllib.request
    req = urllib.request.Request(url, headers={"User-Agent": "subside-backbone-setup/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        data = resp.read()
    dest.write_bytes(data)
    return dest


def parse_ucsd_net(path):
    '''Parse the UCSD Pajek .net file. Returns list of vertex dicts.'''
    vertices = []
    section = None
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:
                continue
            lower = line.lower()
            if lower.startswith("*vertices"):
                section = "vertices"; continue
            if lower.startswith("*edges") or lower.startswith("*arcs"):
                section = "edges"; continue
            if section != "vertices":
                continue
            m = _VERTEX_RE.match(line)
            if not m:
                continue
            vertices.append({
                "id":     int(m.group(1)),
                "label":  m.group(2),
                "x":      float(m.group(3)),
                "y":      float(m.group(4)),
                "x_fact": float(m.group(5)) if m.group(5) else None,
                "y_fact": float(m.group(6)) if m.group(6) else None,
                "color":  m.group(7) or "",
            })
    return vertices


# Reusable stopwords for seeding subdiscipline term lists from their labels
_LABEL_STOP = {"the","and","of","in","on","for","to","with","by","from","an","a",
               "or","general","other","sciences","science","research","studies"}


def build_ucsd_full_backbone(vertices, scaffold_palette=UCSD_SCAFFOLD):
    '''Bucket subdisciplines under their parent discipline via the ic color tag.

    Strategy:
      1. Find the 13 discipline-named vertices; record (color → discipline_name).
      2. Group every remaining vertex by its color → its parent discipline.
      3. Seed each subdiscipline's term list from its label tokens.
      4. Inherit the publication-attested hex color from UCSD_SCAFFOLD when
         the Pajek color string is a plain name (Red/Blue/Green/…).
    '''
    # Step 1: discover discipline → color
    discipline_color = {}     # discipline_name → pajek_color
    discipline_coords = {}    # discipline_name → (x, y)
    for v in vertices:
        if v["label"] in UCSD_DISCIPLINE_NAMES:
            # Normalize minor case/spelling variants to canonical names
            canon = v["label"]
            for d in scaffold_palette:
                if d.lower().replace(" ", "") == canon.lower().replace(" ", ""):
                    canon = d
                    break
            discipline_color[canon] = v["color"]
            discipline_coords[canon] = (v["x"], v["y"])

    if not discipline_color:
        raise RuntimeError("No discipline-named anchor nodes found in .net file. "
                           "The file format may have changed.")

    color_to_discipline = {c: d for d, c in discipline_color.items()}

    # Step 2: initialize empty backbone with metadata inherited from §4 scaffold
    backbone = {}
    for d, pajek_color in discipline_color.items():
        scaffold_node = scaffold_palette.get(d, {})
        hex_color = scaffold_node.get("color")    # publication hex
        backbone[d] = {
            "subdisciplines":      [],
            "terms":               list(scaffold_node.get("terms", [])),
            "color":               hex_color or pajek_color,
            "pajek_color":         pajek_color,
            "coords":              {"x": discipline_coords[d][0],
                                    "y": discipline_coords[d][1]},
            "subdiscipline_coords": {},
            "_source": ("UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-SA 3.0), "
                        "full 554-subdiscipline classification via Science Integrity Alliance"),
        }

    # Step 3: bucket non-discipline vertices
    n_assigned, n_orphan = 0, 0
    for v in vertices:
        if v["label"] in UCSD_DISCIPLINE_NAMES:
            continue
        parent = color_to_discipline.get(v["color"])
        if not parent:
            n_orphan += 1
            continue
        n_assigned += 1
        sub = v["label"]
        backbone[parent]["subdisciplines"].append(sub)
        backbone[parent]["subdiscipline_coords"][sub] = {"x": v["x"], "y": v["y"]}
        # Seed terms from the subdiscipline label
        words = re.findall(r"[A-Za-z][A-Za-z\-]+", sub.lower())
        terms = backbone[parent]["terms"]
        existing = set(terms)
        for w in words:
            if len(w) > 2 and w not in _LABEL_STOP and w not in existing:
                terms.append(w); existing.add(w)
        sub_l = sub.lower()
        if sub_l not in existing:
            terms.append(sub_l)

    # Sort subdisciplines alphabetically within each discipline for readability
    for d in backbone:
        order = sorted(range(len(backbone[d]["subdisciplines"])),
                       key=lambda i: backbone[d]["subdisciplines"][i].lower())
        backbone[d]["subdisciplines"] = [backbone[d]["subdisciplines"][i] for i in order]

    return backbone, n_assigned, n_orphan


# Try to use the full classification; fall back to the compact scaffold on failure
if USE_UCSD_FULL:
    try:
        net_path = fetch_ucsd_net()
        size_kb = net_path.stat().st_size / 1024
        print(f"✓ Pajek source: {net_path}  ({size_kb:.1f} KB)")
        vertices = parse_ucsd_net(net_path)
        print(f"✓ Parsed {len(vertices)} vertices "
              f"(expected ≈ 567 = 554 subdisciplines + 13 disciplines)")
        layer_B_full, n_assigned, n_orphan = build_ucsd_full_backbone(vertices)
        if n_orphan:
            print(f"  ⚠ {n_orphan} vertices had a color that didn't match any discipline anchor; "
                  f"they were dropped.")
        # Replace layer B with the full-grain version
        layer_B = layer_B_full
        n_total_subs = sum(len(node["subdisciplines"]) for node in layer_B.values())
        print(f"✓ Layer B upgraded: {len(layer_B)} disciplines × "
              f"{n_total_subs} subdisciplines ({n_assigned} assigned)")
        for d, node in layer_B.items():
            print(f"   · {d:50s} {len(node['subdisciplines']):3d} subs · "
                  f"{len(node['terms']):4d} terms · {node['pajek_color']}")
    except Exception as e:
        print(f"  ⚠ Could not load the full UCSD classification: {type(e).__name__}: {e}")
        print(f"  Falling back to the compact 13-discipline scaffold from §4.")
        print(f"  To retry: download")
        print(f"    {UCSD_NET_URL}")
        print(f"  manually and save it to {UCSD_NET_LOCAL}, then re-run this cell.")
else:
    print("USE_UCSD_FULL is False — keeping the compact 13-discipline scaffold from §4.")


✓ Pajek source: /scratch/01813/sawp33/tapis/b673a5f6-e828-4ff9-add8-fd125f967b4a-007/work/dso_cookbook_fixes/ScienceBackboneResults/UCSDmap_with_disciplines.net.txt  (103.0 KB)
✓ Parsed 567 vertices (expected ≈ 567 = 554 subdisciplines + 13 disciplines)
  ⚠ 24 vertices had a color that didn't match any discipline anchor; they were dropped.
✓ Layer B upgraded: 12 disciplines × 531 subdisciplines (531 assigned)
   · Biology                                             43 subs ·   88 terms · OliveGreen
   · Biotechnology                                       11 subs ·   33 terms · Emerald
   · Medical Specialties                                 69 subs ·  128 terms · Red
   · Chemical, Mechanical, & Civil Engineering           76 subs ·  186 terms · SkyBlue
   · Chemistry                                           32 subs ·   73 terms · Blue
   · Earth Sciences                                      22 subs ·   72 terms · Mahogany
   · Electrical Engineering & Computer Science           57 su

## 5. Combine the layers

Strategy:

- **Both Layer A and Layer B present** → produce *both* in the JSON file under
  separate keys (`"eto"` and `"ucsd"`), plus a `"merged"` view that overlays
  ETO domains/fields on top of the UCSD scaffold's keyword sets. The main
  notebook can pick whichever it wants via the `backbone_layer` setting it
  exposes after Tier 0.
- **Only Layer B present** → produce just `"ucsd"` and tag `"merged"` to point
  at it. This is the most common case until you do the ETO export.
- **Only Layer A present** → mirror it into `"merged"`.

The merge rule is conservative: where Layer A discipline names match a UCSD
discipline (case-insensitive contains), we *extend* the UCSD term set with
Layer A's `key_concepts`/`topics`. ETO domains that don't match anything in
UCSD are added as new top-level entries so nothing is dropped.


In [14]:
def merge_layers(layer_A, layer_B):
    '''Return a merged dict. Layer B is the scaffold, Layer A extends it.'''
    if not layer_B and not layer_A:
        return None
    if not layer_A:
        return {d: dict(node, _source="UCSD canonical only") for d, node in layer_B.items()}
    if not layer_B:
        return {d: dict(node, _source="ETO only") for d, node in layer_A.items()}

    merged = {d: {"subdisciplines": list(node["subdisciplines"]),
                  "terms":          list(node["terms"]),
                  "color":          node.get("color"),
                  "_source":        "UCSD canonical (extended by ETO)"}
              for d, node in layer_B.items()}

    # Match ETO domains to UCSD on case-insensitive substring overlap
    ucsd_lower = {d.lower(): d for d in merged}
    for eto_d, eto_node in layer_A.items():
        match = None
        eto_lower = eto_d.lower()
        # Try exact
        if eto_lower in ucsd_lower:
            match = ucsd_lower[eto_lower]
        else:
            # Try token overlap (>= 1 substantive token shared)
            eto_tokens = {t for t in re.split(r"\W+", eto_lower) if len(t) > 3}
            for u_lower, u_d in ucsd_lower.items():
                u_tokens = {t for t in re.split(r"\W+", u_lower) if len(t) > 3}
                if eto_tokens & u_tokens:
                    match = u_d
                    break
        if match:
            # Extend UCSD discipline
            existing_subs  = set(merged[match]["subdisciplines"])
            existing_terms = set(merged[match]["terms"])
            for s in eto_node["subdisciplines"]:
                if s not in existing_subs:
                    merged[match]["subdisciplines"].append(s)
                    existing_subs.add(s)
            for t in eto_node["terms"]:
                t_l = t.lower()
                if t_l not in existing_terms:
                    merged[match]["terms"].append(t_l)
                    existing_terms.add(t_l)
            merged[match]["_source"] = "UCSD canonical + ETO match: " + eto_d
        else:
            # Add ETO domain as a new entry
            merged[eto_d] = dict(eto_node)
            merged[eto_d]["_source"] = "ETO only (no UCSD match)"

    return merged


backbone_eto    = layer_A
backbone_ucsd   = layer_B
backbone_merged = merge_layers(layer_A, layer_B)

print(f"✓ Merged: {len(backbone_merged)} domains")
print(f"   Layer A (ETO)  : {len(layer_A) if layer_A else 0} domains")
print(f"   Layer B (UCSD) : {len(layer_B)} disciplines")


✓ Merged: 12 domains
   Layer A (ETO)  : 0 domains
   Layer B (UCSD) : 12 disciplines


## 6. Persist the backbone to `science_backbone.json`

The main notebook's Tier 0 hook reads this file. The schema is:

```json
{
  "schema_version": "1.0",
  "produced_at":    "<ISO timestamp>",
  "produced_by":    "SUBSIDE_ScienceBackbone_Setup.ipynb",
  "selected_layer": "merged" | "ucsd" | "eto",
  "citations":      [...],
  "layers": {
    "eto":    { ... } | null,
    "ucsd":   { ... },
    "merged": { ... }
  }
}
```

`selected_layer` is the one the main notebook will load by default; you can
change it later by editing the JSON in place.


In [15]:
# Default selection: prefer 'merged' when both layers exist, else 'ucsd'
SELECTED_LAYER = ("merged" if (layer_A and layer_B) else
                  "ucsd"   if layer_B            else
                  "eto"    if layer_A            else None)

CITATIONS = [
    {
        "label": "UCSD Map of Science",
        "citation": ("Börner K, Klavans R, Patek M, Zoss AM, Biberstine JR, "
                     "Light RP, Larivière V, Boyack KW (2012). Design and "
                     "Update of a Classification System: The UCSD Map of "
                     "Science. PLOS ONE 7(7): e39464."),
        "doi": "10.1371/journal.pone.0039464",
        "license": "CC BY-NC-SA 3.0",
        "attribution_text": ("The authors wish to acknowledge The Regents of "
                             "the University of California, SciTech Strategies, "
                             "Observatoire des Sciences et des Technologies, "
                             "and the Cyberinfrastructure for Network Science "
                             "Center for making the 2010 UCSD Map of Science "
                             "and Classification System available for this work."),
    },
    {
        "label": "Consensus Map of Science",
        "citation": ("Klavans R, Boyack KW (2009). Toward a Consensus Map of "
                     "Science. JASIST 60(3): 455–476."),
        "doi": "10.1002/asi.20991",
    },
    {
        "label": "Mapping Knowledge Domains",
        "citation": ("Shiffrin RM, Börner K (2004). Mapping Knowledge Domains. "
                     "PNAS 101 (Suppl. 1): 5183–5185."),
        "doi": "10.1073/pnas.0307852100",
    },
    {
        "label": "ETO Map of Science",
        "citation": ("Emerging Technology Observatory, Center for Security "
                     "and Emerging Technology, Georgetown University. "
                     "ETO Map of Science."),
        "url": "https://sciencemap.eto.tech/",
        "methodology_url": "https://eto.tech/dataset-docs/mac-clusters/",
    },
]

payload = {
    "schema_version": "1.0",
    "produced_at":    datetime.utcnow().isoformat() + "Z",
    "produced_by":    "SUBSIDE_ScienceBackbone_Setup.ipynb",
    "bib_source":     str(BIB_PATH),
    "eto_csv_source": str(ETO_CSV_PATH) if ETO_CSV_PATH else None,
    "selected_layer": SELECTED_LAYER,
    "citations":      CITATIONS,
    "layers": {
        "eto":    backbone_eto,
        "ucsd":   backbone_ucsd,
        "merged": backbone_merged,
    },
    "search_seeds_for_eto_query":   seeds if 'seeds' in dir() else [],
}

with open(BACKBONE_JSON_PATH, "w") as fh:
    json.dump(payload, fh, indent=2)

print(f"✓ Wrote {BACKBONE_JSON_PATH}")
print(f"  Selected layer: {SELECTED_LAYER}")
print(f"  File size     : {BACKBONE_JSON_PATH.stat().st_size:,} bytes")


✓ Wrote /scratch/01813/sawp33/tapis/b673a5f6-e828-4ff9-add8-fd125f967b4a-007/work/dso_cookbook_fixes/ScienceBackboneResults/science_backbone.json
  Selected layer: ucsd
  File size     : 215,581 bytes


## 7. Diagnostic view — verify before handing off

Quick summary of what the main notebook will see. Use this to spot
mis-mapped ETO domains, sparse UCSD term sets, or duplicated subdisciplines
before running downstream analyses.


In [16]:
def summarize_backbone(backbone, label):
    print(f"\n── {label} ──")
    if not backbone:
        print("  (empty)")
        return
    for d, node in backbone.items():
        n_subs  = len(node.get("subdisciplines", []))
        n_terms = len(node.get("terms", []))
        src     = node.get("_source", "")
        print(f"  {d:55s} subs={n_subs:2d}  terms={n_terms:3d}  [{src[:55]}]")

summarize_backbone(backbone_ucsd,   "Layer B — UCSD canonical")
summarize_backbone(backbone_eto,    "Layer A — ETO (may be empty)")
summarize_backbone(backbone_merged, f"MERGED ({SELECTED_LAYER} will be used by default)")



── Layer B — UCSD canonical ──
  Biology                                                 subs=43  terms= 88  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Biotechnology                                           subs=11  terms= 33  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Medical Specialties                                     subs=69  terms=128  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Chemical, Mechanical, & Civil Engineering               subs=76  terms=186  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Chemistry                                               subs=32  terms= 73  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Earth Sciences                                          subs=22  terms= 72  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Electrical Engineering & Computer Science               subs=57  terms=142  [UCSD Map of Science 2010 (Börner et al. 2012, CC BY-NC-]
  Brain Research

## 8. Wiring this into `SUBSIDE_Science_Backbone.ipynb`

The main notebook already has a **Tier 0** hook (added in v1.1) that loads the
JSON this notebook just wrote. To use it:

1. **Open** `SUBSIDE_Science_Backbone.ipynb`.
2. In §1.1, set:

   ```python
   SCIENCE_BACKBONE_JSON = "<absolute path to science_backbone.json>"
   ```

   If you've followed the default paths, that's
   `ScienceBackboneResults/science_backbone.json` relative to this notebook.
3. **Run** the main notebook normally. §5 prints `Tier 0: …` followed by the
   selected layer name in its source line.

If `SCIENCE_BACKBONE_JSON` is left `None` (the default), the main notebook
behaves exactly as it did before — Tier 1 (ETO via sbp) → Tier 2 (sbp default)
→ Tier 3 (SUBSIDE inline).

### Refreshing the backbone

Re-run **this** notebook only when:

- You've done a fresh ETO Map of Science CSV export (point `ETO_CSV_PATH` at it).
- Your corpus has shifted enough that the §2.1 search seeds change appreciably.
- You want to extend the UCSD `_SCAFFOLD` dict with new subdisciplines.

The main notebook can be re-run as often as you like without touching this one.

### Citing the result

When you publish anything derived from this backbone, include:

- **Börner et al. 2012** (UCSD Map of Science) — required attribution per the
  CC BY-NC-SA 3.0 license. The full text is in the JSON under
  `citations[0].attribution_text`.
- **ETO Map of Science** — if Layer A was used. The required citation form is
  the label `"ETO Map of Science"` plus a link to `https://sciencemap.eto.tech/`.
- **Klavans & Boyack 2009** — when discussing the consensus-map motivation.
- **Shiffrin & Börner 2004** — when framing the broader knowledge-domain
  mapping context.

All four are pre-formatted in `payload["citations"]` for easy export to your
manuscript's reference manager.

---

## What this notebook does, at a glance

| Section | Role |
|---|---|
| 1 | Setup, paths, dependency check |
| 2 | Quick BibTeX summary → ETO query seed terms |
| 3 | ETO Map of Science walkthrough + CSV ingest → Layer A backbone |
| 4 | UCSD canonical 13-discipline scaffold (Börner et al. 2012) → Layer B |
| 5 | Merge Layer A + B (UCSD scaffold extended by ETO matches) |
| 6 | Persist `science_backbone.json` with citations + provenance |
| 7 | Diagnostic summary of all three layers |
| 8 | Wiring instructions for the main notebook |

This setup notebook is the *referenced and reusable* anchor for cross-disciplinary
analyses across SUBSIDE — and for any other corpus you'd like to project onto a
science basemap.
